In [ ]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from locations import extract_url_map
from events import parse_mmd_taxonomy, extract_events
from timespan import parse_timespan

In [ ]:
root_path = "../data/schede mappatura/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
}

In [ ]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede

## A list of individual sources for experimentation

ignored in the oveall logic

In [ ]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

After identification of all sources

# Shortlist processable sources

In [ ]:
overview = {
    "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [ ]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = [
            "event_label",
            "event_type",
            "place_name",
            "place_type",
            "place_category",
            "wikidata_qid",
            "geonames_id",
            "google maps",
            "date_certainty",
            "date_label",
            "memorial_inscription",
            "source_doc",
            "source_timecode",
            "source_quote",
            "external_links",
            "notes",
        ]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        # df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)

# Track start/end locations: end_location = current row's place,
# start_location = previous event's place (per person)
start_locations = []
prev_location = {}  # protagonist → last place_name
for _, row in df.iterrows():
    person = row["protagonist"]
    end_loc = row["place_name"].strip()
    start_loc = prev_location.get(person, end_loc)  # fallback to same as end
    start_locations.append(start_loc)
    if end_loc:
        prev_location[person] = end_loc
df["start_location"] = start_locations
df["end_location"] = df["place_name"]

# Collect concepts from event_type, place_type, place_category
concept_labels = set()
for col in ["event_type", "place_type", "place_category"]:
    for val in df[col]:
        v = val.strip()
        if v and v != "nan":
            concept_labels.add(v)
print(f"Concepts to create: {sorted(concept_labels)}")

df


# Locations

In [ ]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip().removesuffix(".0")
        )

    locs[name].update(urls[0])
print(locs)

## Add GO concepts as locations

Concepts under 'GO – Luoghi geografici' are geographic locations.
Add them to the locs dict so they get included in locations.xlsx.

In [ ]:
from taxonomy import load_taxonomy

taxonomy = load_taxonomy()

go_concepts = [
    label for label, cat in taxonomy.get("concept_to_category", {}).items()
    if cat == "GO"
]
# Also include GO sub-category labels
for key, info in taxonomy.get("sub_categories", {}).items():
    if info.get("parent") == "GO":
        go_concepts.append(info["label"])

for concept_name in go_concepts:
    concept_name = concept_name.strip()
    if not concept_name:
        continue
    if concept_name not in locs:
        locs[concept_name] = {}
    if "label" not in locs[concept_name] or not locs[concept_name]["label"]:
        locs[concept_name]["label"] = "GO"

print(f"Added {len(go_concepts)} GO concepts to locations, total: {len(locs)}")


## Update preexisting locations

In [ ]:
import os
import re
from locations import enrich_locations_xlsx

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    rich = pd.read_excel("locations.xlsx", dtype=str)
    rich = rich.set_index(["location"])
else:
    rich = pd.DataFrame()
    rich.index.name = "location"

# Build a normalized index for flexible matching
existing_keys = {_normalize_loc(idx): idx for idx in rich.index}

for location, row in locs.items():
    key = _normalize_loc(location)

    if key in existing_keys:
        # Enrich existing row: only fill absent cells
        real_idx = existing_keys[key]
        if isinstance(row, dict):
            for col, val in row.items():
                if col not in rich.columns:
                    rich[col] = ""
                existing = rich.loc[real_idx, col]
                if isinstance(existing, pd.Series):
                    existing = existing.iloc[0]
                if pd.isna(existing) or str(existing).strip() in ("", "nan"):
                    rich.loc[real_idx, col] = str(val)
    else:
        # New location: add row with provided values
        if isinstance(row, dict):
            for col in row:
                if col not in rich.columns:
                    rich[col] = ""
            rich.loc[location] = {col: str(val) for col, val in row.items()}
        else:
            rich.loc[location] = pd.Series(dtype=str)
        existing_keys[key] = location

rich.to_excel("locations.xlsx")

# Enrich with bag-of-words and super-region columns
enrich_locations_xlsx("locations.xlsx")

# Timespan

In [ ]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

# Notes

left unprocessed for now

In [ ]:
set(df["notes"])

# Links

left unprocessed for now

In [ ]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

# Events

TODO: incomplete due to too much noise. Issues:

- use LL or Lebenslauf, currently extracted as one, but need to be two equivalent
- "alter heimant" instread of "alte heimat"
- "Transport", "Tod des Vaters",  are not label

In [ ]:
event_taxonomy = parse_mmd_taxonomy("../docs/maxqda_coding_taxonomy.mmd")
events = sorted(set(event_taxonomy.keys()), key=lambda x: -len(x))
len(events), events[:5] + ["..."] + events[-5:]

In [ ]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

In [ ]:
import json

from api_client import (
    login, api_post, api_patch,
    clean_str, extract_urls_from_text,
    get_or_create_concept, get_or_create_location,
    get_or_create_timespan, get_or_create_url,
    get_person_id, _concept_cache,
    link_locations_to_regions,
)
from locations import (
    load_locations_db, save_locations_db,
    upsert_location_db, normalize_location, _locations_db,
)
from events import classify_lifecycle

login()
load_locations_db()

# === Main import ===

# 1. Create all concepts
print("Creating concepts...")
for label in sorted(concept_labels):
    get_or_create_concept(label)
print(f"  {len(concept_labels)} concept labels processed")

# 1b. Build concept taxonomy (parent + icon)
print("Setting up concept taxonomy...")
from taxonomy import load_taxonomy
_tax = load_taxonomy()

# Create root categories with icons
_cat_ids = {}  # category key -> concept id
for key, info in _tax["categories"].items():
    cid = get_or_create_concept(info["label"])
    if cid:
        api_patch("concepts", cid, {"icon": info["icon"], "parent": None})
        _cat_ids[key] = cid

# Create sub-categories with parent
_subcat_ids = {}
for key, info in _tax["sub_categories"].items():
    cid = get_or_create_concept(info["label"])
    parent_id = _cat_ids.get(info["parent"])
    if cid and parent_id:
        root_icon = _tax["categories"][info["parent"]]["icon"]
        api_patch("concepts", cid, {"icon": root_icon, "parent": parent_id})
        _subcat_ids[key] = cid

# Set parent + icon on each mapped leaf concept
for label, cat_key in _tax["concept_to_category"].items():
    cid = _concept_cache.get(label.lower().strip())
    if not cid:
        cid = get_or_create_concept(label)
    if cid:
        root_icon = _tax["categories"][cat_key]["icon"]
        parent_id = _cat_ids.get(cat_key)
        api_patch("concepts", cid, {"icon": root_icon, "parent": parent_id})

print(f"  Taxonomy set up: {len(_cat_ids)} roots, {len(_subcat_ids)} sub-cats")

# 1c. Link GO concepts to their corresponding LocationPoints
print("Linking GO concepts to locations...")
errors = []
_go_linked = 0
for label, cat_key in _tax["concept_to_category"].items():
    if cat_key != "GO":
        continue
    cid = _concept_cache.get(label.lower().strip())
    if not cid:
        continue
    loc_id = get_or_create_location(label, locations_db=_locations_db,
                                     normalize_fn=normalize_location)
    if loc_id:
        try:
            api_patch("locations", loc_id, {"concept": cid})
            _go_linked += 1
        except Exception as e:
            errors.append(f"GO link {label}: {e}")
# Also link GO sub-category labels
for key, info in _tax["sub_categories"].items():
    if info.get("parent") != "GO":
        continue
    cid = _concept_cache.get(info["label"].lower().strip())
    if not cid:
        continue
    loc_id = get_or_create_location(info["label"], locations_db=_locations_db,
                                     normalize_fn=normalize_location)
    if loc_id:
        try:
            api_patch("locations", loc_id, {"concept": cid})
            _go_linked += 1
        except Exception as e:
            errors.append(f"GO link {info['label']}: {e}")
print(f"  Linked {_go_linked} GO concepts to locations")


# 2. Import events
print("\nImporting events...")
created_events = []
errors = []
_prev_date = {}  # protagonist -> last date_label (for date propagation)

for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Events")):
    try:
        # Timespan from date_label (propagate from previous row if missing)
        date_label = clean_str(row["date_label"])
        person_key = clean_str(row["protagonist"])
        if not date_label and person_key:
            date_label = _prev_date.get(person_key, "")
        if date_label and person_key:
            _prev_date[person_key] = date_label
        ts = parse_timespan(date_label) if date_label else None
        timespan_id = get_or_create_timespan(ts.start, ts.end, ts.certainty) if ts and ts.start else None

        # Upsert locations into the locations database
        upsert_location_db(
            row["end_location"],
            wikidata_qid=clean_str(row.get("wikidata_qid", "")),
            geonames_id=clean_str(row.get("geonames_id", "")),
        )
        upsert_location_db(row["start_location"])

        # Locations with external IDs
        end_loc_id = get_or_create_location(
            row["end_location"],
            wikidata_qid=clean_str(row.get("wikidata_qid", "")),
            geonames_id=clean_str(row.get("geonames_id", "")),
            locations_db=_locations_db,
            normalize_fn=normalize_location,
        )
        start_loc_id = get_or_create_location(
            row["start_location"],
            locations_db=_locations_db,
            normalize_fn=normalize_location,
        )

        # Person
        person_id = get_person_id(row["protagonist"], row["name"])

        # is_confirmed from date_certainty
        date_certainty = clean_str(row.get("date_certainty", ""))
        is_confirmed = date_certainty.lower() in ("certain", "sicher", "yes", "ja")

        # Event description
        description = clean_str(row["event_label"])

        # Life journey classification: search all relevant columns
        lifecycle = classify_lifecycle(
            row["event_label"],
            row.get("event_type", ""),
            row.get("place_type", ""),
            row.get("place_category", ""),
        )

        # Build event payload
        payload = {
            "description": description,
            "is_confirmed": is_confirmed,
            "lifecycle": lifecycle,
        }
        if timespan_id:
            payload["timespan"] = timespan_id
        if start_loc_id:
            payload["start_location"] = start_loc_id
        if end_loc_id:
            payload["end_location"] = end_loc_id
        if person_id:
            payload["persons"] = [person_id]

        # URLs from external_links
        url_ids = []
        for url_str in extract_urls_from_text(row.get("external_links", "")):
            if "geonames.org" in url_str or "wikidata.org" in url_str:
                continue
            uid = get_or_create_url(url_str)
            if uid:
                url_ids.append(uid)
        if url_ids:
            payload["urls"] = url_ids

        # Concepts from all four columns
        concept_ids = []
        for col in ["event_label", "event_type", "place_type", "place_category"]:
            val = clean_str(row.get(col, ""))
            if val:
                # Split on ">" for hierarchical labels
                for part in val.split(">"):
                    part = part.strip()
                    if part:
                        cid = get_or_create_concept(part)
                        if cid:
                            concept_ids.append(cid)

        if concept_ids:
            payload["concepts"] = concept_ids

        event = api_post("events", payload)
        event_id = event["id"]

        # Extraction to link concepts, source data, and notes
        source_quote = clean_str(row.get("source_quote", ""))
        source_timecode = clean_str(row.get("source_timecode", ""))
        source_doc = clean_str(row.get("source_doc", ""))
        memorial = clean_str(row.get("memorial_inscription", ""))

        # Build extraction notes from memorial_inscription, source_doc
        notes_parts = []
        if memorial:
            notes_parts.append(f"Memorial inscription: {memorial}")
        if source_doc:
            notes_parts.append(f"Source: {source_doc}")
        extraction_notes = "\n".join(notes_parts)

        if concept_ids or source_quote or extraction_notes:
            extraction_payload = {
                "event": event_id,
                "quote": source_quote,
                "timecode": source_timecode,
                "notes": extraction_notes,
            }
            if person_id:
                extraction_payload["people_mentioned"] = person_id
            if concept_ids:
                extraction_payload["concepts"] = concept_ids
            api_post("extractions", extraction_payload)

        created_events.append(event_id)
    except Exception as e:
        errors.append(f"Row {idx}: {e}")

print(f"\nCreated {len(created_events)} events")
if errors:
    print(f"\n{len(errors)} errors:")
    for err in errors[:20]:
        print(f"  {err}")

# Save updated locations database
save_locations_db()
link_locations_to_regions(_locations_db)
